In [1]:
%%capture
%pip install langchain-openai==0.3.10
%pip install langchain==0.3.21
%pip install openai==1.68.2
%pip install langchain-community==0.2.1
%pip install  --upgrade langgraph
%pip install langchain_community==0.3.24

In [2]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_ORG_ID = os.getenv("OPENAI_ORG_ID", "")

# Model Configuration
DEFAULT_MODEL = os.getenv("DEFAULT_MODEL", "gpt-3.5-turbo")
TEMPERATURE = float(os.getenv("TEMPERATURE", "0.7"))
MAX_TOKENS = int(os.getenv("MAX_TOKENS", "500"))
WATSON_URL = os.getenv("IBM_URL_END_POINT")
PROJECT_ID = os.getenv("IBM_PROJECT_ID")
IBM_API_KEY = os.getenv("IBM_API_KEY")
TAVILY_KEY = os.getenv("TAVILY_KEY")

In [3]:
import os
import json
import getpass
from typing import List, Dict
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import ChatOpenAI
from langgraph.graph import END, MessageGraph

/home/samuel/Desktop/ai_applications_coursera/venv/lib/python3.12/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(


In [4]:
from langchain_openai import ChatOpenAI
from langchain_ibm import ChatWatsonx
openai_llm = ChatOpenAI(
    model="gpt-4.1-nano",
    api_key = OPENAI_API_KEY,
)
watsonx_llm = ChatWatsonx(
    model_id="ibm/granite-8b-code-instruct",
    url="https://us-south.ml.cloud.ibm.com",
    project_id=PROJECT_ID,
    apikey=IBM_API_KEY,
)

In [5]:
def _set_if_undefined(var: str) -> None:
    if os.environ.get(var):
        return
    os.environ[var] = getpass.getpass(var)
_set_if_undefined("TAVILY_API_KEY")

In [6]:
tavily_tool=TavilySearchResults(max_results=1)
sample_query = "healthy breakfast recipes"
search_results = tavily_tool.invoke(sample_query)
print(search_results)

[{'title': '60 Healthy Breakfast Ideas - Recipes by Love and Lemons', 'url': 'https://www.loveandlemons.com/healthy-breakfast-ideas/', 'content': 'Below, I share over 60 healthy breakfast recipes, divided into 11 (yes, 11!) categories: oats, eggs, smoothies, bowls, quick breads, pancakes & waffles, breakfast tacos, breakfast cookies, toast, muffins & scones, and bars & balls. Whether you’re someone who craves something savory or sweet first thing in the morning, or whether you like to enjoy breakfast at home or grab it and go, you’re sure to find some healthy breakfast ideas you love.\n\n#### Healthy Breakfast Oats\n\nOats are loaded with fiber, so they’re a great healthy breakfast!', 'score': 0.9125066}]


In [7]:
llm = ChatOpenAI(model="gpt-4.1-nano")
question="Any ideas for a healthy breakfast"
response=llm.invoke(question).content
print(response)

Certainly! Here are some healthy breakfast ideas to start your day right:

1. Greek yogurt with fresh berries and a sprinkle of granola or nuts
2. Overnight oats topped with sliced bananas and a drizzle of honey
3. Scrambled eggs with spinach and tomatoes
4. Whole-grain toast with avocado and a poached egg
5. Smoothie made with your favorite fruits, spinach, Greek yogurt, and a splash of almond milk
6. Cottage cheese with pineapple chunks and chia seeds
7. Banana and almond butter on whole-grain toast

These options are nutritious, balanced, and can be quickly prepared. Let me know if you have specific dietary preferences or restrictions!


In [8]:
question="Any ideas for a healthy breakfast"
response=llm.invoke(question).content
print(response)

Certainly! Here are some healthy breakfast ideas to start your day:

1. Greek Yogurt Parfait: Layer Greek yogurt with fresh berries, granola, and a drizzle of honey for a protein-packed and satisfying meal.

2. Overnight Oats: Combine oats with milk or a dairy-free alternative, chia seeds, and your favorite fruits. Let it sit overnight in the fridge for a quick and nutritious breakfast.

3. Veggie Omelette: Whisk eggs with chopped vegetables like spinach, tomatoes, and bell peppers. Cook until fluffy for a low-calorie, high-protein option.

4. Smoothie Bowl: Blend your favorite fruits with some spinach or kale, add a splash of almond milk, and top with nuts, seeds, and sliced fruit.

5. Whole Grain Toast with Avocado: Spread ripe avocado on whole grain toast, sprinkle with a little salt, pepper, and optional chili flakes for added flavor.

6. Fresh Fruit Salad: Mix a variety of seasonal fruits for a hydrating and vitamin-rich start to your day.

7. Cottage Cheese with Fruit and Nuts: A

In [9]:
prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are Dr. Paul Saladino, "Carnivore MD," advocating for animal-based nutrition and challenging plant-centric dietary dogma. Focus on the superior bioavailability of animal foods and the potential toxicity of plant compounds such as oxalates, lectins, and phytates.

        Your response must follow these steps:
        1. {first_instruction}
        2. Present the evolutionary and biochemical rationale for animal-based nutrition, emphasizing nutrient density, bioavailability, and the absence of plant antinutrients.
        3. Challenge conventional "plants are healthy" narratives with mechanistic evidence about plant defense compounds and their impact on human physiology.
        4. Reflect and critique your answer. Be rigorous about evolutionary consistency and nutrient science.
        5. After the reflection, **list 1-3 search queries separately** for researching antinutrients, bioavailability studies, or evolutionary nutrition. Do not include them inside the reflection.

        Focus on: organ meats as superfoods, the carnivore elimination protocol, and why "fiber is unnecessary" based on human evolutionary biology.
        """
    ),
    MessagesPlaceholder(variable_name="messages"),
    (
        "system", 
        "Answer the user's question above using the required format, emphasizing the superiority of animal-based nutrition."
    ),
])

In [10]:
first_responder_prompt = prompt_template.partial(first_instruction="Provide a detailed ~250 word answer")
temp_chain = first_responder_prompt| llm
response = temp_chain.invoke({"messages": [HumanMessage(content=question)]})
print(response.content)

From an evolutionary perspective, humans have thrived on diets rich in animal foods, especially organ meats, which are nature’s most concentrated sources of essential nutrients. Organ meats like liver, kidney, and heart provide bioavailable vitamins (such as B12, A, and folate), minerals (zinc, copper, selenium), and unique compounds like coenzyme Q10, all in forms easily absorbed by the human body. Unlike plant sources, these nutrients are not hindered by antinutrients like phytates, lectins, or oxalates that can impair mineral absorption or cause inflammation.

The biochemical advantage of animal-based foods lies in their high bioavailability. Animal proteins contain complete amino acid profiles necessary for tissue repair and enzyme function. Fats from animal sources deliver fat-soluble vitamins (A, D, E, K) and essential omega-3 fatty acids—crucial for brain function—absent in plant foods. Additionally, humans do not require dietary fiber for gut health; instead, the gut is evoluti

In [11]:
class Reflection(BaseModel):
	missing: str = Field(description="What information is missing")
	superfluous: str = Field(description="What information is unnecessary")

class AnswerQuestion(BaseModel):
	answer: str = Field(description="Main response to the question")
	reflection: Reflection = Field(description="Self-critique of the answer")
	search_queries: List[str] = Field(description="Queries for additional research")

In [12]:
initial_chain = first_responder_prompt| llm.bind_tools(tools=[AnswerQuestion])
response=initial_chain.invoke({"messages":[HumanMessage(question)]})
print("---Full Structured Output---")
print(response.tool_calls)

---Full Structured Output---
[{'name': 'AnswerQuestion', 'args': {'answer': 'A highly nutrient-dense and bioavailable breakfast can be centered around animal products such as eggs, organ meats like liver, and high-quality meats. Eggs provide a complete amino acid profile and contain choline, B vitamins, and fat-soluble vitamins in an easily absorbed form. Organ meats, especially liver, are considered superfoods due to their concentration of vitamins A, B12, copper, zinc, and other essential nutrients in their most bioavailable state. These foods deliver nutrients directly to cells, bypassing the digestion and absorption hurdles posed by plant compounds.\n\nIn contrast to plant-based options, animal foods do not contain antinutrients such as oxalates, phytates, and lectins, which can impair mineral absorption and contribute to inflammation or other health issues. Evolutionarily, humans developed an enzyme system optimized for animals’ nutrient profiles, especially in organ meats, which 

In [13]:
answer_content = response.tool_calls[0]['args']['answer']
print("---Initial Answer---")
print(answer_content)

---Initial Answer---
A highly nutrient-dense and bioavailable breakfast can be centered around animal products such as eggs, organ meats like liver, and high-quality meats. Eggs provide a complete amino acid profile and contain choline, B vitamins, and fat-soluble vitamins in an easily absorbed form. Organ meats, especially liver, are considered superfoods due to their concentration of vitamins A, B12, copper, zinc, and other essential nutrients in their most bioavailable state. These foods deliver nutrients directly to cells, bypassing the digestion and absorption hurdles posed by plant compounds.

In contrast to plant-based options, animal foods do not contain antinutrients such as oxalates, phytates, and lectins, which can impair mineral absorption and contribute to inflammation or other health issues. Evolutionarily, humans developed an enzyme system optimized for animals’ nutrient profiles, especially in organ meats, which are rich in bioavailable nutrients that support optimal bi

In [14]:
Reflection_content = response.tool_calls[0]['args']['reflection']
print("---Reflection Answer---")
print(Reflection_content)

---Reflection Answer---
{'missing': 'Specific studies comparing bioavailability of animal vs. plant nutrients.', 'superfluous': "Details about general plant-based diets or fiber's role in other species."}


In [15]:
search_queries = response.tool_calls[0]['args']['search_queries']
print("---Search Queries---")
print(search_queries)

---Search Queries---
['bioavailability of animal versus plant nutrients', 'evolutionary basis of human diet', 'effects of plant antinutrients on human health']


In [16]:
response_list=[]
response_list.append(HumanMessage(content=question))
response_list.append(response)

In [17]:
tool_call=response.tool_calls[0]
search_queries = tool_call["args"].get("search_queries", [])
print(search_queries)

['bioavailability of animal versus plant nutrients', 'evolutionary basis of human diet', 'effects of plant antinutrients on human health']


In [18]:
tavily_tool=TavilySearchResults(max_results=3)



def execute_tools(state: List[BaseMessage]) -> List[BaseMessage]:
    last_ai_message = state[-1]
    tool_messages = []
    for tool_call in last_ai_message.tool_calls:
        if tool_call["name"] in ["AnswerQuestion", "ReviseAnswer"]:
            call_id = tool_call["id"]
            search_queries = tool_call["args"].get("search_queries", [])
            query_results = {}
            for query in search_queries:
                result = tavily_tool.invoke(query)
                query_results[query] = result
            tool_messages.append(ToolMessage(
                content=json.dumps(query_results),
                tool_call_id=call_id)
            )
    return tool_messages

In [20]:
tool_response = execute_tools(response_list)
# Use .extend() to add all tool messages from the list
response_list.extend(tool_response)

In [21]:
tool_response

[ToolMessage(content='{"bioavailability of animal versus plant nutrients": [{"title": "Comparative bioavailability of vitamins in human foods sourced from ...", "url": "https://pubmed.ncbi.nlm.nih.gov/37522617/", "content": "Vitamins are essential components of enzyme systems involved in normal growth and function. The quantitative estimation of the proportion of dietary vitamins, that is in a form available for utilization by the human body, is limited and fragmentary. This review provides the current state of knowledge on the bioavailability of thirteen vitamins and choline, to evaluate whether there are differences in vitamin bioavailability when human foods are sourced from animals or plants. The bioavailability of naturally occurring choline, vitamin D, vitamin E, and vitamin K in food awaits further studies. Animal-sourced foods are the almost exclusive natural sources of dietary vitamin B-12 (65% bioavailable) and preformed vitamin A retinol (74% bioavailable), and contain highl

In [22]:
response_list

[HumanMessage(content='Any ideas for a healthy breakfast', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_Kmmcj9Py00PCmhyBjdGSwAKJ', 'function': {'arguments': '{"answer": "A highly nutrient-dense and bioavailable breakfast can be centered around animal products such as eggs, organ meats like liver, and high-quality meats. Eggs provide a complete amino acid profile and contain choline, B vitamins, and fat-soluble vitamins in an easily absorbed form. Organ meats, especially liver, are considered superfoods due to their concentration of vitamins A, B12, copper, zinc, and other essential nutrients in their most bioavailable state. These foods deliver nutrients directly to cells, bypassing the digestion and absorption hurdles posed by plant compounds.\\n\\nIn contrast to plant-based options, animal foods do not contain antinutrients such as oxalates, phytates, and lectins, which can impair mineral absorption and contribute 

In [23]:
revise_instructions = """Revise your previous answer using the new information, applying the rigor and evidence-based approach of Dr. David Attia.
- Incorporate the previous critique to add clinically relevant information, focusing on mechanistic understanding and individual variability.
- You MUST include numerical citations referencing peer-reviewed research, randomized controlled trials, or meta-analyses to ensure medical accuracy.
- Distinguish between correlation and causation, and acknowledge limitations in current research.
- Address potential biomarker considerations (lipid panels, inflammatory markers, and so on) when relevant.
- Add a "References" section to the bottom of your answer (which does not count towards the word limit) in the form of:
- [1] https://example.com
- [2] https://example.com
- Use the previous critique to remove speculation and ensure claims are supported by high-quality evidence. Keep response under 250 words with precision over volume.
- When discussing nutritional interventions, consider metabolic flexibility, insulin sensitivity, and individual response variability.
"""
revisor_prompt = prompt_template.partial(first_instruction=revise_instructions)

In [24]:
class ReviseAnswer(AnswerQuestion):
    """Revise your original answer to your question."""
    references: List[str] = Field(description="Citations motivating your updated answer.")
revisor_chain = revisor_prompt | llm.bind_tools(tools=[ReviseAnswer])

In [25]:
response = revisor_chain.invoke({"messages": response_list})
print("---Revised Answer with References---")
print(response.tool_calls[0]['args'])

---Revised Answer with References---
{'answer': 'A nutrient-dense, bioavailable, and evolutionarily consistent breakfast includes eggs and organ meats such as liver or kidney. Eggs are rich in complete proteins, choline, B vitamins, and healthy fats, all in highly bioavailable forms. Organ meats, especially liver, provide concentrated sources of vitamin A, B12, iron, zinc, and copper—nutrients that are more bioavailable than their plant sources (e.g., heme iron, preformed vitamin A) [1]. Unlike plants, which contain antinutrients like oxalates, phytates, and lectins that impair mineral absorption and may cause inflammation [2], animal foods lack these compounds. Human evolutionary biology shows our enzyme systems are optimized for animal nutrients, particularly from organs—our ancestral superfoods—that support metabolic flexibility and insulin sensitivity [3]. The myth that dietary fiber from plants is necessary is dissonant with human evolution, as primitive diets were predominantly a

In [26]:
response_list.append(response)

In [28]:
MAX_ITERATIONS = 4
def event_loop(state: List[BaseMessage]) -> str:
    count_tool_visits = sum(isinstance(item, ToolMessage) for item in state)
    num_iterations = count_tool_visits
    if num_iterations >= MAX_ITERATIONS:
        return END
    return "execute_tools"

In [29]:
graph=MessageGraph()

graph.add_node("respond", initial_chain)
graph.add_node("execute_tools", execute_tools)
graph.add_node("revisor", revisor_chain)

/tmp/ipykernel_700309/2942375001.py:1: LangGraphDeprecatedSinceV10: MessageGraph is deprecated in LangGraph v1.0.0, to be removed in v2.0.0. Please use StateGraph with a `messages` key instead. Deprecated in LangGraph V1.0 to be removed in V2.0.
  graph=MessageGraph()


In [30]:
graph.add_edge("respond", "execute_tools")
graph.add_edge("execute_tools", "revisor")

In [31]:
graph.add_conditional_edges("revisor", event_loop)
graph.set_entry_point("respond")

In [32]:
app = graph.compile()
responses = app.invoke(
    """I'm pre-diabetic and need to lower my blood sugar, and I have heart issues.
    What breakfast foods should I eat and avoid"""
)

In [33]:
print("--- Initial Draft Answer ---")
initial_answer = responses[1].tool_calls[0]['args']['answer']
print(initial_answer)
print("\n")

print("--- Intermediate and Final Revised Answers ---")
answers = []

# Loop through all messages in reverse to find all tool_calls with answers
for msg in reversed(responses):
    if getattr(msg, 'tool_calls', None):
        for tool_call in msg.tool_calls:
            answer = tool_call.get('args', {}).get('answer')
            if answer:
                answers.append(answer)

# Print all collected answers
for i, ans in enumerate(answers):
    label = "Final Revised Answer" if i == 0 else f"Intermediate Step {len(answers) - i}"
    print(f"{label}:\n{ans}\n")


--- Initial Draft Answer ---
For someone with pre-diabetes and heart issues, an animal-based breakfast focusing on nutrient-dense, bioavailable foods is optimal. Prioritize eggs, especially yolks rich in choline and healthy fats, and include organ meats like liver, which are rich in vitamins A, B12, and minerals essential for cardiovascular and metabolic health. Beef, lamb, or fatty fish like salmon provide high-quality protein and omega-3 fatty acids, proven to support heart health and improve insulin sensitivity. Eliminating processed carbs, sugars, grains, and plant-based starches reduces glycemic spikes. Intramuscular fats from animal tissues are easier to metabolize than plant-based carbohydrates. The carnivore elimination protocol suggests that excluding plant toxins, such as oxalates, lectins, and phytates, minimizes inflammation and oxidative stress, improving overall metabolic function.


--- Intermediate and Final Revised Answers ---
Final Revised Answer:
For pre-diabetes and